In [1]:
from langchain_core.documents import Document

# 准备测试数据 ，假设我们提供的文档数据如下：
documents = [
    Document(
        page_content="狗是伟大的伴侣，以其忠诚和友好而闻名。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
    Document(
        page_content="猫是独立的宠物，通常喜欢自己的空间。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
    Document(
        page_content="金鱼是初学者的流行宠物，需要相对简单的护理。",
        metadata={"source": "鱼类宠物文档"},
    ),
    Document(
        page_content="鹦鹉是聪明的鸟类，能够模仿人类的语言。",
        metadata={"source": "鸟类宠物文档"},
    ),
    Document(
        page_content="兔子是社交动物，需要足够的空间跳跃。",
        metadata={"source": "哺乳动物宠物文档"},
    ),
]


In [3]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 实例化一个向量数据库
vector_store = Chroma.from_documents(documents, embedding=OpenAIEmbeddings(
    model='Qwen/Qwen3-Embedding-8B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
))

In [5]:
vector_store.similarity_search_with_score('小兔子')

[(Document(id='e38c5360-73f3-4804-bbda-f4e25b3b07f0', metadata={'source': '哺乳动物宠物文档'}, page_content='兔子是社交动物，需要足够的空间跳跃。'),
  1.1212503910064697),
 (Document(id='e3c4daac-6792-4368-abd5-430bbaea3011', metadata={'source': '鸟类宠物文档'}, page_content='鹦鹉是聪明的鸟类，能够模仿人类的语言。'),
  1.1228773593902588),
 (Document(id='5af4b876-660b-4964-82ad-887017892b48', metadata={'source': '哺乳动物宠物文档'}, page_content='狗是伟大的伴侣，以其忠诚和友好而闻名。'),
  1.1456849575042725),
 (Document(id='e22fda06-92cd-4463-9057-b5018d265a43', metadata={'source': '鱼类宠物文档'}, page_content='金鱼是初学者的流行宠物，需要相对简单的护理。'),
  1.1469554901123047)]

In [6]:
from langchain_core.runnables import RunnableLambda

# 根据 vector_store 创建一个检索器  bind(k=1) 表示只返回最相似的1个结果
retriever = RunnableLambda(vector_store.similarity_search).bind(k=1)

In [8]:
retriever.batch(["小兔子","人类的"])

[[Document(id='e38c5360-73f3-4804-bbda-f4e25b3b07f0', metadata={'source': '哺乳动物宠物文档'}, page_content='兔子是社交动物，需要足够的空间跳跃。')],
 [Document(id='e3c4daac-6792-4368-abd5-430bbaea3011', metadata={'source': '鸟类宠物文档'}, page_content='鹦鹉是聪明的鸟类，能够模仿人类的语言。')]]